# SIMBAC-NFS — Interactive Demo

This notebook runs **SIMBAC-NFS** end-to-end on the **Yacht Hydrodynamics** dataset (UCI ID 243).
No local data files needed — data is fetched live from UCI.

1. Load data from UCI
2. Build the pool (T rounds × M models per round)
3. Compress with SIMBA into a single compact model
4. RMSE, MAE, R²
5. IF-THEN rule base printout
6. Membership function visualization
7. Interpretability summary

## 1. Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

sys.path.insert(0, ".")

from src.models.pool import FCMTSKPool
from src.models.compression import SIMBACCompressor

print("Imports OK")

## 2. Load Data

**Yacht Hydrodynamics** — 308 samples, 6 features, predicts residuary resistance of sailing yachts.

In [ ]:
repo = fetch_ucirepo(id=243)
X_df = repo.data.features
y_ser = repo.data.targets

feature_names = X_df.columns.tolist()
X = X_df.values.astype(float)
y = y_ser.values.flatten().astype(float)

print(f"Samples : {len(X)}")
print(f"Features: {feature_names}")
print(f"Target  : residuary_resistance  [{y.min():.2f}, {y.max():.2f}]")

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

## 4. Phase 1 — Build the Pool

T=5 rounds, M=3 FCM-TSK models per round → 15 models total.  
Each round trains on the residuals left by the previous round.

In [ ]:
T  = 5     # rounds
M  = 3     # models per round
LR = 0.3   # learning rate
NC = 15    # rule budget per model

y_res = y_train.copy().astype(float)
pool  = []

for t in range(T):
    round_models = FCMTSKPool(
        n_estimators=M, n_rules=NC, min_rules=3,
        base_params={"mf_type": "gaussian"},
        random_state=42 + t,
    )
    round_models.fit(X_train, y_res)
    round_preds = np.mean([e.predict(X_train) for e in round_models.estimators_], axis=0)
    y_res = y_res - LR * round_preds
    pool.extend(round_models.estimators_)
    print(f"  Round {t+1}: residual RMSE = {np.sqrt(np.mean(y_res**2)):.4f}")

print(f"\nPool: {len(pool)} models, {sum(m.n_rules for m in pool)} rules total")

compressor = SIMBACCompressor(
    tau=0.95,
    weight_by_performance=True,
    cumulative_importance=0.99,
    min_rules=3,
    refit_consequents=True,
    tune_refit_alpha=True,
)

model = compressor.compress(pool, X_train, y_train)

meta = model.compression_meta_
print(f"Pool rules   : {meta['n_pool_rules']}")
print(f"Compressed to: {meta['n_rules_after']} rules")
print(f"Compression  : {meta['compression_pct']}")

In [ ]:
compressor = SIMBACCompressor(
    tau=0.95,
    weight_by_performance=True,
    cumulative_importance=0.99,
    min_rules=3,
    refit_consequents=True,
    tune_refit_alpha=True,
)

model = compressor.compress(pool, X_train, y_train)

meta = model.compression_meta_
print(f"Pool rules   : {meta['n_pool_rules']}")
print(f"Compressed to: {meta['n_rules_after']} rules")
print(f"Compression  : {meta['compression_pct']}")

## 6. Performance

In [ ]:
y_pred = model.predict(X_test)

rmse = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))
mae  = float(np.mean(np.abs(y_test - y_pred)))
r2   = float(1 - np.sum((y_test - y_pred) ** 2)
             / (np.sum((y_test - np.mean(y_test)) ** 2) + 1e-12))

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

## 7. Fuzzy Rule Base

In [ ]:
print(f"=== SIMBAC-NFS Rule Base ({model.n_rules} rules) ===\n")
for i, rule in enumerate(model.get_linguistic_labels(feature_names)):
    antecedent = " AND ".join(f"{feat} is {lbl}" for feat, lbl in rule.items())
    print(f"Rule {i+1:2d}: IF {antecedent}")

## 8. Membership Functions

In [ ]:
n_feat = len(feature_names)
cols   = min(n_feat, 3)
rows   = (n_feat + cols - 1) // cols
colors = plt.cm.tab10(np.linspace(0, 1, model.n_rules))
x_vals = np.linspace(0, 1, 300)

fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3.0 * rows), squeeze=False)

for j, fname in enumerate(feature_names):
    ax = axes[j // cols][j % cols]
    for r in range(model.n_rules):
        c = float(model.centers_[r, j])
        s = float(max(model.sigmas_[r, j], 1e-3))
        ax.plot(x_vals, np.exp(-0.5 * ((x_vals - c) / s) ** 2),
                color=colors[r], alpha=0.75, linewidth=1.5)
    ax.set_title(fname, fontsize=9, fontweight="bold")
    ax.set_xlabel("Normalized value", fontsize=8)
    ax.set_ylabel("Firing strength", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.08)
    ax.tick_params(labelsize=7)

for j in range(n_feat, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

plt.suptitle(f"SIMBAC-NFS — {model.n_rules} rules (Yacht Hydrodynamics)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 9. Prediction vs Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.4, s=40)
lim = [min(y_test.min(), y_pred.min()) - 1, max(y_test.max(), y_pred.max()) + 1]
ax.plot(lim, lim, "r--", linewidth=1.2, label="Perfect fit")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title(f"Predicted vs Actual  (R² = {r2:.4f})")
ax.legend()

ax = axes[1]
ax.scatter(y_pred, y_test - y_pred, alpha=0.6, edgecolors="k", linewidths=0.4, s=40)
ax.axhline(0, color="r", linestyle="--", linewidth=1.2)
ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
ax.set_title("Residuals")

plt.suptitle("SIMBAC-NFS — Yacht Hydrodynamics", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
pool_rules  = meta["n_pool_rules"]
final_rules = meta["n_rules_after"]

print("=" * 42)
print("       SIMBAC-NFS Summary")
print("=" * 42)
print(f"  Pool models              : {len(pool):>6}")
print(f"  Pool rules               : {pool_rules:>6}")
print(f"  Rules after compression  : {final_rules:>6}")
print(f"  Rule reduction           : {100*(1-final_rules/pool_rules):>5.1f}%")
print("-" * 42)
print(f"  Test RMSE                : {rmse:>8.4f}")
print(f"  Test MAE                 : {mae:>8.4f}")
print(f"  Test R²                  : {r2:>8.4f}")
print("=" * 42)